# Inventaire et profilage initial des données

**Objectif :** comprendre le contenu et la structure des fichiers clients et produits, puis repérer les premiers problèmes à examiner.

Cette étape consiste à :
- Présenter les fichiers : nombre de lignes, colonnes et types de données.
- Identifier les clés candidates et les champs potentiellement obligatoires ou uniques.
- Repérer les valeurs manquantes, les espaces superflus et les différences de format.
- Décrire les colonnes numériques à l’aide de statistiques.
- Examiner les catégories et leurs fréquences.
- Vérifier les formats des principales informations.

Les valeurs d’origine sont conservées. Aucune correction n’est appliquée à cette étape.

In [1]:
import pandas as pd

clients_raw = pd.read_csv(
    "../data/raw/clients.csv",
    sep=";",
    encoding="utf-8-sig",
    dtype=str,
    keep_default_na=False,
)

products_raw = pd.read_csv(
    "../data/raw/produits.csv",
    sep=";",
    encoding="utf-8-sig",
    dtype=str,
    keep_default_na=False,
)

clients_df = clients_raw.copy()
products_df = products_raw.copy()

## 1. Présentation des fichiers clients et produits

In [2]:
def describe_dataset(df):
    print(f"Rows: {df.shape[0]}")
    print(f"Columns: {df.shape[1]}")

    display(df.dtypes.rename("loaded_type").to_frame())
    display(df.head())

In [3]:
# Clients dataset overview
describe_dataset(clients_df)

Rows: 10
Columns: 11


,loaded_type
id_client,str
prenom,str
nom,str
age,str
email,str
telephone,str
ville,str
code_postal,str
type_client,str
date_inscription,str


,id_client,prenom,nom,age,email,telephone,ville,code_postal,type_client,date_inscription,entreprise_exemple
0,CLI001,Camille,Leroy,29,camille.leroy@example.com,06 12 34 50 01,Lille,59000,Particulier,2025-02-14,Les Saveurs du Nord (entreprise fictive)
1,CLI001,Thomas,Dubois,41,thomas.dubois@example.com,06 12 34 50 02,Roubaix,59100,Professionnel,2024-11-06,Les Saveurs du Nord (entreprise fictive)
2,CLI003,Inès,Moreau,34,,06 12 34 50 03,Tourcoing,59200,Particulier,2025-05-22,Les Saveurs du Nord (entreprise fictive)
3,CLI004,Hugo,Petit,46,hugo.petit@@example,06 12 34 50 04,Villeneuve-d'Ascq,59650,Professionnel,2024-09-18,Les Saveurs du Nord (entreprise fictive)
4,CLI005,Sarah,Rousseau,150,sarah.rousseau@example.com,06 12 34 50 05,Lambersart,59130,Particulier,2025-01-09,Les Saveurs du Nord (entreprise fictive)


In [4]:
# Products dataset overview
describe_dataset(products_df)

Rows: 100
Columns: 10


,loaded_type
id_produit,str
nom_produit,str
categorie,str
conditionnement,str
prix_unitaire_eur,str
tva_pct,str
stock_unites,str
statut_stock,str
type_fournisseur,str
entreprise_exemple,str


,id_produit,nom_produit,categorie,conditionnement,prix_unitaire_eur,tva_pct,stock_unites,statut_stock,type_fournisseur,entreprise_exemple
0,PRD001,Gaufres artisanales,Biscuits & douceurs,paquet 6 pièces,4.2,5.5,36.0,En stock,Fournisseur alimentaire,Les Saveurs du Nord (entreprise fictive)
1,PRD001,Spéculoos pur beurre,Biscuits & douceurs,sachet 250 g,3.6,5.5,70.0,En stock,Artisan partenaire,Les Saveurs du Nord (entreprise fictive)
2,PRD003,Bêtises de Cambrai à la menthe,Confiserie,boîte 200 g,5.9,5.5,34.0,En stock,Atelier local,Les Saveurs du Nord (entreprise fictive)
3,PRD004,Caramels au beurre salé,Confiserie,sachet 150 g,4.8,5.5,16.0,En stock,Producteur régional,Les Saveurs du Nord (entreprise fictive)
4,PRD005,Ch'ti pain d'épices,,pain 300 g,5.5,5.5,63.0,En stock,Producteur régional,Les Saveurs du Nord (entreprise fictive)


## 2. Identification des clés candidates et des contraintes attendues

### Fichier clients

- `id_client` est l’identifiant proposé : il devrait être obligatoire
  et unique pour chaque client. Ces conditions restent à vérifier.
- Les autres champs obligatoires dépendent des besoins métier :
  par exemple, un moyen de contact peut être nécessaire pour contacter
  le client.
- L’e-mail et le téléphone ne sont pas considérés comme uniques sans
  confirmation : plusieurs clients peuvent partager les mêmes coordonnées.
- Le nom et le prénom ne permettent pas d’identifier un client de manière
  unique, car plusieurs personnes peuvent porter le même nom.

### Fichier produits

- `id_produit` est l’identifiant proposé : il devrait être obligatoire
  et unique pour chaque produit. Ces conditions restent à vérifier.
- Pour exploiter le catalogue commercial, les champs `nom_produit`,
  `categorie`, `prix_unitaire_eur`, `tva_pct` et `stock_unites` sont
  proposés comme obligatoires, sous réserve de validation métier.
- Le nom du produit n’est pas supposé unique : des produits peuvent
  avoir un nom identique et des conditionnements différents.

### Points à vérifier

Les identifiants proposés ne pourront être confirmés comme clés candidates
que si les contrôles montrent qu’ils sont renseignés et uniques.

Les autres obligations doivent être confirmées par les règles métier,
et non déduites uniquement du contenu actuel des fichiers.

## 3. Valeurs manquantes, espaces et différences de format

In [5]:
def check_missing_and_spaces(df):
    summary = pd.DataFrame({
        "null_count": df.isna().sum(),
        "empty_string_count": df.eq("").sum(),
        "whitespace_only_count": df.apply(
            lambda column: column.str.fullmatch(r"\s+", na=False).sum()
        ),
        "surrounding_spaces_count": df.apply(
            lambda column: (
                column.notna() & column.ne(column.str.strip())
            ).sum()
        ),
    })

    display(summary)

def inspect_values(df, columns):
    for column in columns:
        print(f"\nColumn: {column}")
        print(f"Distinct values: {df[column].nunique(dropna=False)}")

        display(
            df[column]
            .map(repr)
            .value_counts(dropna=False)
            .rename("count")
            .to_frame()
        )

In [6]:
# Check missing values and spaces in the clients dataset
check_missing_and_spaces(clients_df)

,null_count,empty_string_count,whitespace_only_count,surrounding_spaces_count
id_client,0,0,0,0
prenom,0,0,0,0
nom,0,0,0,0
age,0,0,0,0
email,0,1,0,0
telephone,0,0,0,0
ville,0,0,0,0
code_postal,0,0,0,0
type_client,0,0,0,0
date_inscription,0,0,0,0


**Observation :** un e-mail est absent (10 % des clients). Aucun espace
superflu n’est détecté.

In [7]:
# Inspect spelling, capitalization and date formats
inspect_values(
    clients_df,
    ["ville", "type_client", "date_inscription", "code_postal", "telephone"],
)


Column: ville
Distinct values: 10


,count
ville,
'Lille',1
'Roubaix',1
'Tourcoing',1
"""Villeneuve-d'Ascq""",1
'Lambersart',1
'Marcq-en-Barœul',1
'Armentières',1
'Lens',1
'Douai',1



Column: type_client
Distinct values: 2


,count
type_client,
'Particulier',5
'Professionnel',5



Column: date_inscription
Distinct values: 10


,count
date_inscription,
'2025-02-14',1
'2024-11-06',1
'2025-05-22',1
'2024-09-18',1
'2025-01-09',1
'2025-03-30',1
'2024-12-12',1
'2025-06-03',1
'2025-04-17',1



Column: code_postal
Distinct values: 10


,count
code_postal,
'59000',1
'59100',1
'59200',1
'59650',1
'59130',1
'59700',1
'59280',1
'75001',1
'59500',1



Column: telephone
Distinct values: 10


,count
telephone,
'06 12 34 50 01',1
'06 12 34 50 02',1
'06 12 34 50 03',1
'06 12 34 50 04',1
'06 12 34 50 05',1
'06 12',1
'06 12 34 50 07',1
'06 12 34 50 08',1
'06 12 34 50 09',1


**Observation :** le téléphone `06 12` semble incomplet.
Les clients se répartissent entre 5 particuliers et 5 professionnels.

In [8]:
# Check missing values and spaces in the products dataset
check_missing_and_spaces(products_df)

,null_count,empty_string_count,whitespace_only_count,surrounding_spaces_count
id_produit,0,0,0,0
nom_produit,0,0,0,0
categorie,0,1,0,0
conditionnement,0,0,0,0
prix_unitaire_eur,0,0,0,0
tva_pct,0,0,0,0
stock_unites,0,1,0,0
statut_stock,0,0,0,0
type_fournisseur,0,0,0,0
entreprise_exemple,0,0,0,0


**Observation :** une catégorie et une quantité en stock sont absentes
(1 % chacune). Aucun espace superflu n’est détecté.

In [9]:
# Inspect category labels and numerical formats
inspect_values(
    products_df,
    ["categorie", "conditionnement", "statut_stock",
     "prix_unitaire_eur", "tva_pct", "stock_unites"],
)


Column: categorie
Distinct values: 21


,count
categorie,
'Épicerie sucrée',13
'Boissons',9
'Apéritif',7
'Confiserie',6
'Farines & céréales',6
'Coffrets & kits',6
'Biscuits & douceurs',5
'Boissons chaudes',5
'Huiles & condiments',5



Column: conditionnement
Distinct values: 55


,count
conditionnement,
'sachet 500 g',8
'sachet 150 g',6
'bouteille 1 L',5
'bouteille 33 cl',4
'bocal 180 g',4
'sachet 100 g',3
'pot 320 g',3
'sachet 1 kg',3
'tablette 100 g',3



Column: statut_stock
Distinct values: 2


,count
statut_stock,
'En stock',74
'Stock faible',26


... (2 outputs left out)


Column: tva_pct
Distinct values: 3


,count
tva_pct,
'5.5',90
'20.0',9
'99.0',1



Column: stock_unites
Distinct values: 71


,count
stock_unites,
'36.0',3
'22.0',3
'48.0',3
'132.0',3
'75.0',3
...,...
'115.0',1
'60.0',1
'152.0',1


**Observation :** le prix de -5,20 €, le prix de 9 999,90 € et la TVA
de 99 % sont à vérifier. Les 21 valeurs distinctes de catégorie incluent
une valeur vide, soit 20 catégories renseignées.

## 4. Statistiques descriptives des colonnes numériques

In [10]:
def describe_numeric_columns(df, columns):
    numeric_df = df[columns].apply(pd.to_numeric, errors="coerce")

    # Report nonblank values that could not be converted
    for column in columns:
        populated = df[column].str.strip().ne("")
        failed = populated & numeric_df[column].isna()

        print(f"{column}: {failed.sum()} nonblank conversion failures")

    display(
        numeric_df.describe().T.rename(
            columns={"50%": "median"}
        )
    )

In [11]:
# Describe client ages
describe_numeric_columns(clients_df, ["age"])

age: 0 nonblank conversion failures


,count,mean,std,min,25%,median,75%,max
age,10.0,60.3,55.073789,0.0,35.0,42.5,48.25,172.0


**Observation :** les âges vont de 0 à 172 ans. La moyenne (60,3 ans)
dépasse la médiane (42,5 ans), notamment sous l’effet des valeurs extrêmes
à vérifier.

In [12]:
# Describe prices, VAT rates and stock quantities
describe_numeric_columns(
    products_df,
    ["prix_unitaire_eur", "tva_pct", "stock_unites"],
)

prix_unitaire_eur: 0 nonblank conversion failures
tva_pct: 0 nonblank conversion failures
stock_unites: 0 nonblank conversion failures


,count,mean,std,min,25%,median,75%,max
prix_unitaire_eur,100.0,106.936000,999.325043,-5.2,3.8,4.85,6.425,9999.9
tva_pct,100.0,7.740000,10.116863,5.5,5.5,5.50,5.500,99.0
stock_unites,99.0,79.424242,45.773642,-12.0,36.0,75.00,116.000,158.0


**Observations :**
- Le prix extrême tire la moyenne à 106,94 €, contre une médiane de 4,85 €.
- Un stock de -12 unités est à vérifier.
- Les statistiques du stock portent sur 99 valeurs, car une cellule est vide.

## 5. Description des colonnes catégorielles

Les nombres de valeurs distinctes et les fréquences des catégories sont
présentés dans les résultats de la question 3 pour :

- Clients : `ville` et `type_client`.
- Produits : `categorie`, `conditionnement` et `statut_stock`.

La colonne `type_fournisseur` est examinée en complément dans cette section.

Les valeurs vides et les éventuelles valeurs manquantes restent affichées.
Les différences d’écriture sont conservées et comptées séparément.

In [13]:
# Inspect supplier types and their frequencies
inspect_values(products_df, ["type_fournisseur"])


Column: type_fournisseur
Distinct values: 4


,count
type_fournisseur,
'Fournisseur alimentaire',29
'Artisan partenaire',28
'Producteur régional',27
'Atelier local',16


## 6. Vérification des formats

Vérifier les formats des dates, codes postaux, e-mails, téléphones,
prix, taux de TVA et quantités.

Règles initiales :
- Dates : format YYYY-MM-DD et date existante.
- Codes postaux français : cinq chiffres.
- E-mails : structure simple de type nom@domaine.extension.
- Téléphones français : format national ou international avec +33.
- Prix et taux : valeurs convertibles en nombres.
- Quantités : valeurs convertibles en nombres entiers.

Ces contrôles vérifient les formats, sans garantir l’exactitude des données.
Les cellules vides sont suivies séparément à la question 3.

In [14]:
def show_format_issues(df, column, valid):
    populated = df[column].notna() & df[column].str.strip().ne("")
    invalid = populated & ~valid.fillna(False)

    print(f"{column}: {invalid.sum()} nonblank values to review")
    display(df.loc[invalid, [df.columns[0], column]])

In [15]:
# Check dates using the expected format
dates = pd.to_datetime(
    clients_df["date_inscription"],
    format="%Y-%m-%d",
    errors="coerce",
)

valid_dates = (
    clients_df["date_inscription"].str.fullmatch(
        r"[0-9]{4}-[0-9]{2}-[0-9]{2}", na=False
    )
    & dates.notna()
)

show_format_issues(clients_df, "date_inscription", valid_dates)

date_inscription: 0 nonblank values to review


,id_client,date_inscription


In [16]:
# Check French postal codes
valid_postal_codes = clients_df["code_postal"].str.fullmatch(
    r"[0-9]{5}", na=False
)

show_format_issues(clients_df, "code_postal", valid_postal_codes)

code_postal: 0 nonblank values to review


,id_client,code_postal


**Observation :** tous les codes postaux comportent cinq chiffres.
Leur correspondance avec les villes reste à vérifier.

In [17]:
# Check basic email structure
valid_emails = clients_df["email"].str.fullmatch(
    r"[^@\s]+@[^@\s]+\.[^@\s]+", na=False
)

show_format_issues(clients_df, "email", valid_emails)

email: 1 nonblank values to review


,id_client,email
3,CLI004,hugo.petit@@example


**Observation :** l’e-mail `hugo.petit@@example` de `CLI004` est mal
formé. L’e-mail absent est traité séparément dans les valeurs manquantes.

In [18]:
# Check French phone numbers without changing original values
phones = clients_df["telephone"].str.replace(
    r"[\s.\-]", "", regex=True
)

valid_phones = phones.str.fullmatch(
    r"(?:0[1-9][0-9]{8}|\+33[1-9][0-9]{8})",
    na=False,
)

show_format_issues(clients_df, "telephone", valid_phones)

telephone: 1 nonblank values to review


,id_client,telephone
5,CLI006,06 12


**Observation :** le téléphone `06 12` de `CLI006` ne respecte pas
le format attendu.

In [19]:
# Check whether prices and VAT rates can be read as numbers
for column in ["prix_unitaire_eur", "tva_pct"]:
    numbers = pd.to_numeric(products_df[column], errors="coerce")

    show_format_issues(
        products_df,
        column,
        numbers.notna(),
    )

prix_unitaire_eur: 0 nonblank values to review


,id_produit,prix_unitaire_eur


tva_pct: 0 nonblank values to review


,id_produit,tva_pct


**Observation :** toutes les valeurs sont convertibles en nombres,
mais cela ne garantit pas leur validité métier.

In [20]:
# Check whether stock quantities are whole numbers
stock = pd.to_numeric(products_df["stock_unites"], errors="coerce")

valid_stock = stock.notna() & stock.mod(1).eq(0)

show_format_issues(products_df, "stock_unites", valid_stock)

stock_unites: 0 nonblank values to review


,id_produit,stock_unites


**Observation :** les 99 stocks renseignés sont des nombres entiers.
Le stock négatif passe ce contrôle de format, mais reste à vérifier.